# Data Engineering and Management WorkshopThis workbook contains the worked solution for the workshop. Use it to check the method, the code, and the reasoning behind each step.The case study follows a fictional prosthetics charity and shows the core parts of a data engineering workflow: ingestion, cleaning, joins, and privacy masking.You will work through four themes:- Ingestion and pipelines- Data cleaning and logic checks- Relational storage and joins- Cybersecurity and privacy maskingBefore you start, run the mock data generation script so that `messy_prosthetics_data.csv` is in the same folder as this notebook.

## Step 0: Mock Data SetupRun the generator script first. It creates `messy_prosthetics_data.csv` and the small reference table used later in the workbook.

## Section 1: Ingestion & Pipelines (ETL vs ELT)ETL means **Extract, Transform, Load**. Data is validated before it reaches the destination system. In a healthcare setting, that usually reduces risk because privacy checks and validation happen before sensitive data moves on.ELT means **Extract, Load, Transform**. Raw data lands first and is transformed later in a warehouse or lakehouse. That works well when you need scale and can separate sensitive records from direct identifiers.In practice, a charity may use ETL for privacy-critical workflows and ELT for reporting once the data has been safely separated from direct identifiers.

In [ ]:
from pathlib import Path

import pandas as pd

csv_candidates = [
    Path('messy_prosthetics_data.csv'),
    Path('..') / 'messy_prosthetics_data.csv',
    Path.cwd() / 'messy_prosthetics_data.csv',
]

for candidate in csv_candidates:
    if candidate.exists():
        csv_path = candidate
        break
else:
    raise FileNotFoundError('Could not find messy_prosthetics_data.csv. Run the mock data script first.')

df = pd.read_csv(csv_path)
df.info()

## Section 2: Data Cleaning & Logic ChallengesData validation checks whether values make sense before they enter downstream systems. Type cleaning and standardisation keep messy inputs from breaking joins, reports, or database loads.

In [ ]:
import re

CURRENT_YEAR = 2026

def parse_age_input(value):
    if pd.isna(value):
        return None

    if isinstance(value, (int, float)) and not pd.isna(value):
        numeric_value = int(value)
    else:
        match = re.search(r'\d+', str(value))
        if not match:
            return None
        numeric_value = int(match.group())

    if numeric_value < 100:
        return numeric_value
    if numeric_value > 1900:
        return CURRENT_YEAR - numeric_value
    return numeric_value

df['Age_Years'] = df['Age_Input'].apply(parse_age_input)
df[['Patient_ID', 'Age_Input', 'Age_Years']]

In [ ]:
df['Implant_Date_Clean'] = pd.to_datetime(df['Implant_Date'], errors='coerce', dayfirst=True).dt.strftime('%Y-%m-%d')
df['Amputation_Cause_Clean'] = df['Amputation_Cause'].str.lower().str.strip()

df[['Patient_ID', 'Implant_Date', 'Implant_Date_Clean', 'Amputation_Cause', 'Amputation_Cause_Clean']]

## Section 3: Relational Storage & Big DataStructured data fits into tables, rows, and columns, so SQL databases are a good fit when you need rules and relationships. Unstructured data includes notes, messages, and scanned documents.Primary keys identify rows. Foreign keys point to related rows in other tables. Joins use those keys to connect data without duplicating it.For this charity, patient records can be joined to device inventory so analysts can see which model was fitted, what it cost, and who made it.

In [ ]:
device_inventory = {
    'M210': {'Model_Name': 'AeroStep Lite', 'Cost_USD': 950, 'Manufacturer': 'Stride MedTech'},
    'M302': {'Model_Name': 'FlexFoot Alpha', 'Cost_USD': 1200, 'Manufacturer': 'Global Limb Works'},
    'M404': {'Model_Name': 'CrestFit Pro', 'Cost_USD': 1850, 'Manufacturer': 'OrthoSphere'},
    'M505': {'Model_Name': 'AdaptMax Field', 'Cost_USD': 2100, 'Manufacturer': 'Humanity Mobility Systems'},
}

device_inventory_df = pd.DataFrame.from_dict(device_inventory, orient='index').reset_index()
device_inventory_df = device_inventory_df.rename(columns={'index': 'Device_Model_ID'})
device_inventory_df = device_inventory_df[['Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]

patient_device_df = pd.merge(df, device_inventory_df, on='Device_Model_ID', how='inner')
patient_device_df[['Patient_ID', 'Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]

## Section 4: Cybersecurity & Cloud Privacy (PII Masking)PII means Personally Identifiable Information. Names, contact details, and other identifiers should be protected, minimised, or removed before data is shared or uploaded.Before patient data is moved to public cloud storage, it should be anonymised or pseudonymised. Hashing can replace names with stable tokens that do not expose the original value.

In [ ]:
import hashlib

def anonymize_name(value):
    normalized_value = str(value).strip().lower()
    return hashlib.sha256(normalized_value.encode('utf-8')).hexdigest()

df['Full_Name'] = df['Full_Name'].apply(anonymize_name)
df[['Patient_ID', 'Full_Name']]

## Wrap-UpBy the end of the exercises, you should be able to explain how messy inputs are handled, how joins connect business tables, and why privacy controls matter before cloud upload.